<small><font color=gray>Notebook author: <a href="https://www.linkedin.com/in/olegmelnikov/" target="_blank">Oleg Melnikov</a>, <a href="https://www.hse.ru/en/staff/sara/" target="_blank">Saraa Ali</a>  ©2026 onwards</font></small><hr style="margin:0;background-color:silver">

**[<font size=6>🧬Genomics</font>](https://www.kaggle.com/t/ade6fdb522504517990bd8b862e50928)**. [**Instructions**](https://colab.research.google.com/drive/1owkYjuRGkx050LQnM3b3yTzd0Dr2XbeV) for running Colabs.

<small>**(Optional) CONSENT.** <mark>[ X ]</mark> We consent to sharing our Colab (after the assignment ends) with other students/instructors for educational purposes. We understand that sharing is optional and this decision will not affect our grade in any way. <font color=gray><i>(If ok with sharing your Colab for educational purposes, leave "X" in the check box.)</i></font></small>

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')   # OK to enable, if your kaggle.json is stored in Google Drive

Mounted at /content/drive


In [33]:
!pip -q install --upgrade --force-reinstall --no-deps kaggle > log  # upgrade kaggle package (to avoid a warning)
!mkdir -p ~/.kaggle                               # .kaggle folder must contain kaggle.json for kaggle executable to properly authenticate you to Kaggle.com
!cp /content/drive/MyDrive/kaggle.json ~/.kaggle/kaggle.json >log  # First, download kaggle.json from kaggle.com (in Account page) and place it in the root of mounted Google Drive
!cp kaggle.json ~/.kaggle/kaggle.json > log       # Alternative location of kaggle.json (without a connection to Google Drive)
!chmod 600 ~/.kaggle/kaggle.json                  # give only the owner full read/write access to kaggle.json
!kaggle config set -n competition -v 26-hse-genomics # set the competition context for the next few kaggle API calls. !kaggle config view - shows current settings
!kaggle competitions download >> log              # download competition dataset as a zip file
!unzip -o *.zip >> log                            # Kaggle dataset is copied as a single file and needs to be unzipped.
!kaggle competitions leaderboard --show           # print public leaderboard

zsh:1: command not found: pip
cp: /content/drive/MyDrive/kaggle.json: No such file or directory
cp: kaggle.json: No such file or directory
- competition is now set to: 26-hse-genomics
Using competition: 26-hse-genomics
Next Page Token = CfDJ8CS0IeAoHcJGgSEc27rBbk6EHTxKtMrU91oyN9VCFFF-7TPW5cTpYR-bnin2i-yJSF7SrY7G7QB2isPITWrtQZo
  teamId  teamName                       submissionDate              score    
--------  -----------------------------  --------------------------  -------  
15564287  L-oony Legacy                  2026-04-17 19:44:11.380000  0.98800  
15589041  H-hhhh                         2026-04-12 08:42:36.310000  0.98687  
15570418  Y-fidat                        2026-04-17 16:05:40.183000  0.98687  
15590663  BL                             2026-04-17 15:56:06.560000  0.98675  
15564267  D-esert Eagle                  2026-04-09 18:18:40.636000  0.98662  
15604189  BF – No, Ain't Got One         2026-04-12 21:27:29.793000  0.98662  
15586776  AK                           

In [34]:
!nvidia-smi --query-gpu=gpu_name,memory.total,memory.free,memory.used --format=csv # GPU specs

zsh:1: command not found: nvidia-smi


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [35]:
%%time
%%capture
%reset -f
!pip -q install -U sentence-transformers > log
from IPython.core.interactiveshell import InteractiveShell as IS; IS.ast_node_interactivity = "all"
import numpy as np, pandas as pd, time, matplotlib.pyplot as plt, os, plotly
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer as SBERT
from sklearn.svm import SVC, LinearSVC, NuSVC
ToCSV = lambda df, fname: df.round(2).to_csv(f'{fname}.csv', index_label='id') # rounds values to 2 decimals

class Timer():
  def __init__(self, lim:'RunTimeLimit'=60*5): self.t0, self.lim, _ = time.time(), lim, print(f'⏳ started. You have {lim} sec. Good luck!')
  def ShowTime(self):
    msg = f'Runtime is {time.time()-self.t0:.0f} sec'
    print(f'\033[91m\033[1m' + msg + f' > {self.lim} sec limit!!!\033[0m' if (time.time()-self.t0-1) > self.lim else msg)

np.set_printoptions(linewidth=10000, precision=4, edgeitems=20, suppress=True)
pd.set_option('display.max_rows', 100, 'display.max_columns', 100, 'display.max_colwidth', 100, 'display.precision', 2, 'display.max_rows', 4)

CPU times: user 231 ms, sys: 306 ms, total: 537 ms
Wall time: 1.85 s


In [36]:
vX = pd.read_csv('testX/testX.csv').set_index('id')
tYX = pd.read_csv('trainYX/trainYX.csv').set_index('id')
vX

,DNA
id,
100000,TTGATTAATAAGATTCCTTGACACCCTTTGTAAAGTTTCTATTTCGTGTGAAATATCTATCTCTTCAAATCCTTTTAATTTATCTAGGTATTTGCT...
100001,ATTAGTAACGGAGGATTTACTAGATGTTTGGATTTATATTCTAATTTTATTCAGGTGGAAGGGATTGTTTTATGATTCAATAGTATACAGAGAATA...
...,...
119998,CGTCGGCATGCTCGGGCAGTGCGGCGGGCCAGCAGCGTGCCAGTTGTCGCGGGGCGGCCGGGCATCGCGGCGCCGGGCGGCAGCACTCCCGCGAAG...
119999,GCGAGGGCACGAAGGCACGACGGCAACGGCGGCGAGGAGCGCTGTGGCAACCGTCTCCGCGTTTGCGTGCGTACAGCCGAGAGCTGGTTCGCGCAG...


In [37]:
tmr = Timer() # runtime limit (in seconds). Add all of your code after the timer

⏳ started. You have 300 sec. Good luck!


❗Do not modify the setup above.

<hr color=red>

<font size=5>⏳</font> <strong><font color=orange size=5>Your Code, Documentation, Ideas and Timer - All Start Here...</font></strong>

Students: Keep all your definitions, code, documentation **between** ⏳ symbols.

In [ ]:
from itertools import product as iproduct
from sklearn.preprocessing import StandardScaler


tYX_full = pd.read_csv('trainYX/trainYX.csv').set_index('id')
train_dna = tYX_full.DNA.values
test_dna  = vX.DNA.values



def kmer_matrix(seqs, k):
    kmers = [''.join(p) for p in iproduct('ACGT', repeat=k)]
    lens  = np.array([max(1, len(s) - k + 1) for s in seqs], dtype=np.float32)
    return np.column_stack([
        np.array([s.count(km) for s in seqs], dtype=np.float32) / lens
        for km in kmers
    ])

tK = np.hstack([kmer_matrix(train_dna, k) for k in [1, 2, 3, 4]])
vK = np.hstack([kmer_matrix(test_dna,  k) for k in [1, 2, 3, 4]])



sc = StandardScaler()
tX = sc.fit_transform(tK)
vX_s = sc.transform(vK)


m = LinearSVC(C=0.5, max_iter=20000, random_state=11)
m.fit(tX, tYX_full.Y)
print(f'Train accuracy: {m.score(tX, tYX_full.Y):.4f}')


pY = pd.DataFrame(m.predict(vX_s), index=vX.index, columns=['y'])
ToCSV(pY, 'MySubmission')



,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",0.5
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: int, default=0Enable verbose output. Note that this setting takes advantage of aper-process runtime setting in liblinear that, if enabled, may not workproperly in a multithreaded context.",0
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo rand

Train accuracy: 0.9844


## **Task 1. Preprocessing Pipeline**

Explain elements of your preprocessing pipeline i.e. feature engineering, subsampling, clustering, dimensionality reduction, etc.
1. Why did you choose these elements? (Something in EDA, prior experience,...? Btw, EDA is not required)
1. How do you evaluate the effectiveness of these elements?
1. What else have you tried that worked or didn't?

**Student's answer:**

1. K-mer features (k=1,2,3,4)
DNA is a string of the letters A, C, G, T. To feed it into the model, it needs to be converted into numbers. K-mer calculates the frequency of substrings of length k. Triplets (k=3)—codons that encode amino acids, are especially important. There are 340 numerical features in total for each sequence.
2. Full dataset for training
I decided to train model on a full data (not only 10k like in the baseline) to let it capture more patterns and generalize better.
3. StandardScaler
normalization makes all features of the same scale so that no single feature dominates the others.

Effectiveness of preproccesing methods was assessed at kaggle score and in-sample accuracy.

I also tried to use SBERT embedding and both K-mer + SBERT. 
1. SBERT showed the best result, but it took a lot of time (approximately 8 minutes). I tried to use it not on the full data, but only on a part (15k). Unfortunately, that didn’t help, and the accuracy became lower.
2. The combination of two feature sets without dimensionality reduction performed worse than k-mer alone.

## **Task 2. Modeling Approach**
Explain your modeling approach, i.e. ideas you tried and why you thought they would be helpful.

1. How did these decisions guide you in modeling?
1. How do you evaluate the effectiveness of these elements?
1. What else have you tried that worked or didn't?

**Student's answer:**

From the available models, LinearSVC was chosen because it performs quickly even on a large dataset of 100k samples. A standard SVC model with an RBF kernel would have taken too long to train on such a large dataset, exceeding the 300-seconds time limit.

C=0.5 was selected experimentally. I also tried other values, but accuracy was lower.

[SBERT](https://www.sbert.net) generates 384-dimensional text embedding vectors for each text entry. See [more models](https://www.sbert.net/docs/pretrained_models.html).
* Only reputable publicly available embedding models are allowed (SBERT, USE, MUSE, LASER, ...). We want to prevent participants' training embeddings on test data.

In [ ]:
%%capture
sbert = SBERT('paraphrase-MiniLM-L6-v2')  # 4 seconds; loads SBERT embeddings model

In [ ]:
tYX = tYX.head(10000)
%time tEmb, vEmb = sbert.encode([s[:50] for s in tYX.DNA]), sbert.encode([s[:50] for s in vX.DNA])  # embed all sequences to same-size vectors
print(f'Train embedding matrix size:', tEmb.shape)
pd.DataFrame(vEmb[:3,:30], index=[x[:20]+'...' for x in vX.DNA[:3]]).style.background_gradient(cmap='coolwarm', axis=1).format("{:.2f}") # show movie description and a few of its embedding features

CPU times: user 14.4 s, sys: 272 ms, total: 14.7 s
Wall time: 13.6 s
Train embedding matrix size: (10000, 384)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29
TTGATTAATAAGATTCCTTG...,-0.51,-0.01,-0.09,0.00,-0.61,0.13,0.80,-0.33,0.27,-0.22,0.35,-0.40,-0.02,-0.07,-0.05,0.19,-0.03,0.82,0.10,-0.00,0.46,-0.68,-0.36,-0.08,0.50,0.33,-0.45,0.10,0.70,0.29
ATTAGTAACGGAGGATTTAC...,-0.61,0.03,-0.36,0.26,-0.10,-0.06,0.81,-0.30,0.42,-0.11,0.25,-0.40,0.09,-0.06,-0.23,-0.01,0.30,0.52,0.17,0.21,0.73,-0.39,-0.30,-0.02,0.46,-0.06,-0.80,-0.08,0.37,0.18
AAAAAGCCGTAAAAGACGAT...,-0.30,-0.03,-0.29,0.14,-0.30,0.17,1.02,-0.17,0.19,0.06,0.37,-0.87,-0.23,0.07,-0.14,-0.07,0.04,-0.07,0.17,0.36,0.22,-0.15,-0.41,0.02,0.30,0.36,-0.68,0.09,0.35,0.18


In [ ]:
%time m = LinearSVC(random_state=0, max_iter=10000).fit(tEmb, tYX.Y)  # SVC is ok to use. See updated Rules.
m.score(tEmb, tYX.Y)   # in-sample accuracy

CPU times: user 996 ms, sys: 31.3 ms, total: 1.03 s
Wall time: 1.03 s


0.9507

In [ ]:
#pY = pd.DataFrame(m.predict(vEmb), index=vX.index, columns=['y'])   # predicted targets
#ToCSV((pY>0.5)*1, 'MySubmission')

# **References:**

1. Remember to cite your sources here as well! At the least, your textbook should be cited. Google Scholar allows you to effortlessly copy/paste an APA citation format for books and publications. Also cite StackOverflow, package documentation, and other meaningful internet resources to help your peers learn from these (and to avoid plagiarism claims).
1. ...
1. ...

1. lecture and seminar 9 materials
2. Itertools library documentation https://www.geeksforgeeks.org/python/python-itertools-product/
3. SBERT docs https://sbert.net/
4. To understand how do k-mers work https://medium.com/swlh/bioinformatics-1-k-mer-counting-8c1283a07e29

<font color=green><h4><b>$\epsilon$. LLM Documentation if used</b></h4></font>

<font color=red><b>Your answer here.</b></font>

<font size=5>⌛</font> <strong><font color=orange size=5>Do not exceed competition's runtime limit!</font></strong>

<hr color=red>


In [39]:
tmr.ShowTime()    # measure Colab's runtime. Do not remove. Keep as the last cell in your notebook.

Runtime is 153 sec


# 💡**Starter Ideas**

1. Learn about DNA [&#127910;](https://www.youtube.com/results?search_query=nucleotides+genes+amino+acids+)
1. Try a larger training sample.
1. Try longer training DNA strings, but SBERT may have a cap on string length, so you might split DNA into several strings and then concatenate or average resulting vectors
1. Try other pretrained SBERT models. Note that DNA sequence uses ACGT letters, but many other models were trained on multilingual text. So, you might prefer those that were trained on mostly ASCII.
1. SBERT is trained on word tokens (typically, separated by spaces), but DNA sequence has no spaces. Try placing spaces after every character or some semantically meaningful subsequences (this might require more domain knowledge).
1. Try Google's [USE](https://tfhub.dev/google/universal-sentence-encoder-multilingual/3) embedding models
1. Try Facebook's [LASER](https://github.com/facebookresearch/LASER) and [others](https://tfhub.dev/s?module-type=text-language-model).
1. Try [Enformer](https://tfhub.dev/deepmind/enformer/1) for gene expressions. See [DeepMind paper](https://deepmind.com/blog/article/enformer).
1. Try building your own embeddings on the given sequences. SBERT and other packages make it easy (just a few lines), but it may take too much time.
1. Assess distribution of character patterns (single, doubles, triplets, ...). For example, an ACGT string generates AC, CG, GT doubles and ACG and CGT triplets. Does one class have more subsequences of some type? This might be a feature in your model.
1. Try features built as counts of subsequences (singles, doubles, triplets, ...). Consider EDA first.
1. Concatenate or otherwise combine multiple embeddings derived from each gene string
1. Learn from [*The genetic code*](https://www.khanacademy.org/science/ap-biology/gene-expression-and-regulation/translation/a/the-genetic-code-discovery-and-properties), Khan Academy.
1. Learn from [*Apply Machine Learning Algorithms for Genomics Data Classification*](https://medium.com/mlearning-ai/apply-machine-learning-algorithms-for-genomics-data-classification-132972933723)
1. Learn from [*Efficient counting of k-mers in DNA sequences using a bloom filter*](https://bmcbioinformatics.biomedcentral.com/articles/10.1186/1471-2105-12-333) Páll Melsted et al. 2011
1. Try [Byte Pair Encoding](https://www.derczynski.com/papers/archive/BPE_Gage.pdf) and [SentencePiece](https://arxiv.org/pdf/1808.06226.pdf) to auto identification of "important" [k-mers](https://en.wikipedia.org/wiki/K-mer) (substrings)